# Experiment 1: TF-IDF + Logistic Regression

## Overview
- **Model**: Logistic Regression
- **Vectorizer**: TF-IDF (max_features=10000, ngrams=(1,2))
- **Cross-Validation**: 5-Fold Stratified
- **Approach**: Pipeline-based (prevents data leakage)

## Key Features
✅ Fold-wise TF-IDF fitting (no data leakage)  
✅ Complete pipelines saved (vectorizer + model)  
✅ Averaged predictions from all 5 folds  
✅ Comprehensive metrics tracking  
✅ Reusable pipeline function for future models  

## Output Files
- `pipeline_fold_1.pkl` to `pipeline_fold_5.pkl`
- `metrics.json` (all metrics + best fold info)
- `feature_names_all_folds.json`
- `exp_1_tfidf_log_submission.csv`

In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, log_loss
from sklearn.pipeline import Pipeline
import os
import pickle
import json
from datetime import datetime

# Optuna (for hyperparameter tuning)
import optuna

In [2]:
DATA_PATH = "data/"
EXPERIMENT_PATH = "experiments/"
EXPERIMENT_NAME = "exp_1_tfidf_log"

In [3]:
# Read the data
train = pd.read_csv(DATA_PATH + "train.csv")
test = pd.read_csv(DATA_PATH + "test.csv")

# Deduplicate rows
train = train.drop_duplicates().reset_index(drop=True)

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
print(f"\nTrain columns: {train.columns.tolist()}")
print(f"Test columns: {test.columns.tolist()}")

Train shape: (4212, 2)
Test shape: (1178, 2)

Train columns: ['text', 'label']
Test columns: ['id', 'text']


In [4]:
from sklearn.pipeline import Pipeline

# Create experiment directory
experiment_dir = os.path.join(EXPERIMENT_PATH, EXPERIMENT_NAME)
os.makedirs(experiment_dir, exist_ok=True)
print(f"Experiment directory: {experiment_dir}")

Experiment directory: experiments/exp_1_tfidf_log


In [5]:
def create_pipeline(vectorizer_params=None, model_params=None):
    """
    Create a sklearn pipeline with vectorizer and model.
    This ensures proper fitting on train data only and prevents data leakage.
    
    Args:
        vectorizer_params: dict of TfidfVectorizer parameters
        model_params: dict of model parameters
    
    Returns:
        sklearn.pipeline.Pipeline
    """
    if vectorizer_params is None:
        vectorizer_params = {
            'max_features': 10000,
            'ngram_range': (1, 2),
            'min_df': 2,
            'max_df': 0.95,
            'sublinear_tf': True
        }
    
    if model_params is None:
        model_params = {
            'max_iter': 1000,
            'random_state': 42,
            'solver': 'liblinear'
        }
    
    pipeline = Pipeline([
        ('vectorizer', TfidfVectorizer(**vectorizer_params)),
        ('model', LogisticRegression(**model_params))
    ])
    
    return pipeline

# Initialize pipeline parameters
vectorizer_params = {
    'max_features': 10000,
    'ngram_range': (1, 2),
    'min_df': 2,
    'max_df': 0.95,
    'sublinear_tf': True
}

model_params = {
    'max_iter': 1000,
    'random_state': 42,
    'solver': 'liblinear'
}

print("Pipeline configuration:")
print(f"Vectorizer: TF-IDF with {vectorizer_params}")
print(f"Model: Logistic Regression with {model_params}")

Pipeline configuration:
Vectorizer: TF-IDF with {'max_features': 10000, 'ngram_range': (1, 2), 'min_df': 2, 'max_df': 0.95, 'sublinear_tf': True}
Model: Logistic Regression with {'max_iter': 1000, 'random_state': 42, 'solver': 'liblinear'}


In [6]:
# Prepare data
X_train_text = train['text'].values
y_train = train['label'].values
X_test_text = test['text'].values

print(f"Train samples: {len(X_train_text)}")
print(f"Test samples: {len(X_test_text)}")

Train samples: 4212
Test samples: 1178


In [7]:
# 5-Fold Cross Validation
n_folds = 5
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

# Storage for models and metrics
fold_models = []
fold_metrics = []
fold_predictions = np.zeros(len(test))
all_feature_names = []  # Store feature names from each fold

In [8]:
for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_text, y_train), 1):
    print(f"\nFold {fold}/{n_folds}")
    print("-"*60)
    
    # Split data - IMPORTANT: Split before any fitting
    X_fold_train, X_fold_val = X_train_text[train_idx], X_train_text[val_idx]
    y_fold_train, y_fold_val = y_train[train_idx], y_train[val_idx]
    
    # Create fresh pipeline for this fold
    pipeline = create_pipeline(vectorizer_params, model_params)
    
    # Fit pipeline on training fold only (TF-IDF learns from train only)
    pipeline.fit(X_fold_train, y_fold_train)
    
    # Validation predictions
    y_val_pred = pipeline.predict(X_fold_val)
    y_val_proba = pipeline.predict_proba(X_fold_val)[:, 1]
    
    # Calculate metrics
    metrics = {
        'fold': fold,
        'accuracy': accuracy_score(y_fold_val, y_val_pred),
        'precision': precision_score(y_fold_val, y_val_pred),
        'recall': recall_score(y_fold_val, y_val_pred),
        'f1': f1_score(y_fold_val, y_val_pred),
        'roc_auc': roc_auc_score(y_fold_val, y_val_proba),
        'log_loss': log_loss(y_fold_val, y_val_proba),
        'train_size': len(train_idx),
        'val_size': len(val_idx)
    }
    
    fold_metrics.append(metrics)
    
    # Print metrics
    print(f"Accuracy:  {metrics['accuracy']:.4f}")
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall:    {metrics['recall']:.4f}")
    print(f"F1 Score:  {metrics['f1']:.4f}")
    print(f"ROC AUC:   {metrics['roc_auc']:.4f}")
    print(f"Log Loss:  {metrics['log_loss']:.4f}")
    
    # Test predictions using this fold's pipeline
    test_proba = pipeline.predict_proba(X_test_text)[:, 1]
    fold_predictions += test_proba / n_folds
    
    # Save pipeline (includes both vectorizer and model)
    pipeline_path = os.path.join(experiment_dir, f'pipeline_fold_{fold}.pkl')
    with open(pipeline_path, 'wb') as f:
        pickle.dump(pipeline, f)
    
    # Save feature names from this fold's vectorizer
    feature_names = pipeline.named_steps['vectorizer'].get_feature_names_out().tolist()
    all_feature_names.append({
        'fold': fold,
        'n_features': len(feature_names),
        'features': feature_names
    })
    
    fold_models.append({
        'fold': fold,
        'pipeline_path': pipeline_path,
        'metrics': metrics,
        'n_features': len(feature_names)
    })

print("="*60)
print("Cross Validation Complete!")
print("="*60)


Fold 1/5
------------------------------------------------------------
Accuracy:  0.9419
Precision: 0.9696
Recall:    0.8615
F1 Score:  0.9123
ROC AUC:   0.9933
Log Loss:  0.2457

Fold 2/5
------------------------------------------------------------
Accuracy:  0.9348
Precision: 0.9617
Recall:    0.8480
F1 Score:  0.9013
ROC AUC:   0.9899
Log Loss:  0.2486

Fold 3/5
------------------------------------------------------------
Accuracy:  0.9454
Precision: 0.9663
Recall:    0.8746
F1 Score:  0.9181
ROC AUC:   0.9910
Log Loss:  0.2465

Fold 4/5
------------------------------------------------------------
Accuracy:  0.9418
Precision: 0.9768
Recall:    0.8547
F1 Score:  0.9117
ROC AUC:   0.9935
Log Loss:  0.2410

Fold 5/5
------------------------------------------------------------
Accuracy:  0.9489
Precision: 0.9810
Recall:    0.8716
F1 Score:  0.9231
ROC AUC:   0.9938
Log Loss:  0.2408
Cross Validation Complete!


In [9]:
# Calculate average metrics and identify best fold
metrics_df = pd.DataFrame(fold_metrics)
print("\nPer-Fold Metrics:")
print(metrics_df.to_string(index=False))

print("\n" + "="*60)
print("Average Metrics Across Folds:")
print("-"*60)
for col in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'log_loss']:
    mean_val = metrics_df[col].mean()
    std_val = metrics_df[col].std()
    print(f"{col.upper():12s}: {mean_val:.4f} ± {std_val:.4f}")

# Identify best fold (highest ROC AUC)
best_fold_idx = metrics_df['roc_auc'].idxmax()
best_fold = metrics_df.loc[best_fold_idx, 'fold']
best_roc_auc = metrics_df.loc[best_fold_idx, 'roc_auc']

print(f"BEST FOLD: {int(best_fold)} (ROC AUC: {best_roc_auc:.4f})")


Per-Fold Metrics:
 fold  accuracy  precision   recall       f1  roc_auc  log_loss  train_size  val_size
    1  0.941874   0.969582 0.861486 0.912343 0.993330  0.245656        3369       843
    2  0.934757   0.961686 0.847973 0.901257 0.989945  0.248612        3369       843
    3  0.945368   0.966292 0.874576 0.918149 0.991039  0.246548        3370       842
    4  0.941805   0.976834 0.854730 0.911712 0.993466  0.240955        3370       842
    5  0.948931   0.980989 0.871622 0.923077 0.993819  0.240794        3370       842

Average Metrics Across Folds:
------------------------------------------------------------
ACCURACY    : 0.9425 ± 0.0053
PRECISION   : 0.9711 ± 0.0078
RECALL      : 0.8621 ± 0.0112
F1          : 0.9133 ± 0.0082
ROC_AUC     : 0.9923 ± 0.0017
LOG_LOSS    : 0.2445 ± 0.0035
BEST FOLD: 5 (ROC AUC: 0.9938)


In [10]:
# Save metrics summary
metrics_summary = {
    'experiment_name': EXPERIMENT_NAME,
    'model_type': 'Logistic Regression',
    'vectorizer': 'TF-IDF',
    'vectorizer_params': vectorizer_params,
    'model_params': model_params,
    'n_folds': n_folds,
    'best_fold': int(best_fold),
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'average_metrics': {
        'accuracy': float(metrics_df['accuracy'].mean()),
        'accuracy_std': float(metrics_df['accuracy'].std()),
        'precision': float(metrics_df['precision'].mean()),
        'precision_std': float(metrics_df['precision'].std()),
        'recall': float(metrics_df['recall'].mean()),
        'recall_std': float(metrics_df['recall'].std()),
        'f1': float(metrics_df['f1'].mean()),
        'f1_std': float(metrics_df['f1'].std()),
        'roc_auc': float(metrics_df['roc_auc'].mean()),
        'roc_auc_std': float(metrics_df['roc_auc'].std()),
        'log_loss': float(metrics_df['log_loss'].mean()),
        'log_loss_std': float(metrics_df['log_loss'].std())
    },
    'fold_metrics': fold_metrics,
    'fold_models': fold_models
}

# Save to JSON
with open(os.path.join(experiment_dir, 'metrics.json'), 'w') as f:
    json.dump(metrics_summary, f, indent=2)

print(f"\nMetrics saved to {os.path.join(experiment_dir, 'metrics.json')}")


Metrics saved to experiments/exp_1_tfidf_log/metrics.json


In [11]:
# Create submission file
submission = pd.DataFrame({
    'id': test['id'],
    'probability': fold_predictions
})

submission_path = os.path.join(experiment_dir, f'{EXPERIMENT_NAME}_submission.csv')
submission.to_csv(submission_path, index=False)

print(f"\nSubmission file created: {submission_path}")
print(f"Shape: {submission.shape}")
print(f"\nFirst few predictions:")
print(submission.head(10))


Submission file created: experiments/exp_1_tfidf_log/exp_1_tfidf_log_submission.csv
Shape: (1178, 2)

First few predictions:
   id  probability
0   0     0.165002
1   1     0.090570
2   2     0.079119
3   3     0.160968
4   4     0.193532
5   5     0.079820
6   6     0.064817
7   7     0.706453
8   8     0.652345
9   9     0.261024


In [12]:
# Save all feature names from each fold
feature_names_path = os.path.join(experiment_dir, 'feature_names_all_folds.json')
with open(feature_names_path, 'w') as f:
    json.dump(all_feature_names, f, indent=2)

print(f"\nFeature names from all folds saved to: {feature_names_path}")
print(f"Feature counts per fold:")
for fold_features in all_feature_names:
    print(f"  Fold {fold_features['fold']}: {fold_features['n_features']} features")


Feature names from all folds saved to: experiments/exp_1_tfidf_log/feature_names_all_folds.json
Feature counts per fold:
  Fold 1: 10000 features
  Fold 2: 10000 features
  Fold 3: 10000 features
  Fold 4: 10000 features
  Fold 5: 10000 features


In [13]:
# Summary of saved artifacts
print("EXPERIMENT SUMMARY")
print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Directory: {experiment_dir}")
print(f"\nSaved Artifacts:")
print(f"  - 5 pipeline files (pipeline_fold_1.pkl to pipeline_fold_5.pkl)")
print(f"    Each pipeline contains: TF-IDF vectorizer + Logistic Regression")
print(f"  - metrics.json (detailed metrics for all folds)")
print(f"  - feature_names_all_folds.json (features from each fold)")
print(f"  - {EXPERIMENT_NAME}_submission.csv (averaged predictions)")
print(f"\nBest Model: Fold {int(best_fold)} (ROC AUC: {best_roc_auc:.4f})")
print(f"\nTo load and use a pipeline:")
print(f"  with open('pipeline_fold_{int(best_fold)}.pkl', 'rb') as f:")
print(f"      pipeline = pickle.load(f)")
print(f"  predictions = pipeline.predict_proba(texts)[:, 1]")

EXPERIMENT SUMMARY
Experiment: exp_1_tfidf_log
Directory: experiments/exp_1_tfidf_log

Saved Artifacts:
  - 5 pipeline files (pipeline_fold_1.pkl to pipeline_fold_5.pkl)
    Each pipeline contains: TF-IDF vectorizer + Logistic Regression
  - metrics.json (detailed metrics for all folds)
  - feature_names_all_folds.json (features from each fold)
  - exp_1_tfidf_log_submission.csv (averaged predictions)

Best Model: Fold 5 (ROC AUC: 0.9938)

To load and use a pipeline:
  with open('pipeline_fold_5.pkl', 'rb') as f:
      pipeline = pickle.load(f)
  predictions = pipeline.predict_proba(texts)[:, 1]


## Reusable Pipeline Template

The `create_pipeline()` function can be reused for future experiments. Just modify the vectorizer and model:

```python
# Example: XGBoost with TF-IDF
from xgboost import XGBClassifier

def create_xgb_pipeline(vectorizer_params=None, model_params=None):
    if vectorizer_params is None:
        vectorizer_params = {'max_features': 10000, 'ngram_range': (1, 2)}
    
    if model_params is None:
        model_params = {'n_estimators': 100, 'learning_rate': 0.1, 'random_state': 42}
    
    return Pipeline([
        ('vectorizer', TfidfVectorizer(**vectorizer_params)),
        ('model', XGBClassifier(**model_params))
    ])
```

Key advantages:
- **No data leakage**: TF-IDF fits only on training data per fold
- **Easy to swap models**: Change `LogisticRegression` to any classifier
- **Easy to swap vectorizers**: Change `TfidfVectorizer` to `CountVectorizer`, etc.
- **Serializable**: Save/load entire pipeline as one object
- **Reproducible**: Same pipeline structure across experiments

## Optuna Hyperparameter Tuning (TF-IDF + Logistic Regression)

We'll tune TF-IDF parameters and LogisticRegression regularization using Optuna. The study will use Stratified K-Fold CV and return mean ROC AUC across folds. The best params will be saved and used to retrain a final pipeline on the full training data.

In [14]:
def optuna_objective(trial, X, y, cv_splits=3, random_state=42):
    # TF-IDF hyperparameters
    max_features = trial.suggest_categorical('max_features', [5000, 10000, 20000, None])
    ngram_min = trial.suggest_int('ngram_min', 1, 3)
    ngram_max = trial.suggest_int('ngram_max', ngram_min, 5)
    min_df = trial.suggest_int('min_df', 1, 5)
    max_df = trial.suggest_float('max_df', 0.7, 1.0)
    sublinear_tf = trial.suggest_categorical('sublinear_tf', [True, False])

    # Logistic Regression hyperparameters
    C = trial.suggest_float('C', 1e-4, 10, log=True)
    solver = trial.suggest_categorical('solver', ['liblinear', 'saga'])
    penalty = 'l2' if solver in ['liblinear', 'saga'] else 'l2'

    vectorizer_params = {
        'max_features': max_features,
        'ngram_range': (ngram_min, ngram_max),
        'min_df': min_df,
        'max_df': max_df,
        'sublinear_tf': sublinear_tf
    }
    model_params = {'C': C, 'solver': solver, 'max_iter': 1000, 'random_state': random_state}

    skf_local = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=random_state)
    roc_aucs = []

    for train_idx, val_idx in skf_local.split(X, y):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        # Create and fit pipeline
        pipeline = Pipeline([
            ('vectorizer', TfidfVectorizer(**{k: v for k, v in vectorizer_params.items() if v is not None})),
            ('model', LogisticRegression(**model_params))
        ])

        pipeline.fit(X_tr, y_tr)
        y_val_proba = pipeline.predict_proba(X_val)[:, 1]
        roc_aucs.append(roc_auc_score(y_val, y_val_proba))

        # pruning (optional)
        trial.report(np.mean(roc_aucs), len(roc_aucs))
        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(roc_aucs))

In [15]:
# Run Optuna study (small number of trials by default; increase for production)
study_path = os.path.join(experiment_dir, 'optuna_study.pkl')

study = optuna.create_study(direction='maximize', study_name=EXPERIMENT_NAME)
try:
    study.optimize(lambda trial: optuna_objective(trial, X_train_text, y_train, cv_splits=3), n_trials=50)
except KeyboardInterrupt:
    print('Study interrupted by user')

print(f"Best trial value: {study.best_value}")
print(f"Best trial params: {study.best_params}")

# Save study
with open(study_path, 'wb') as f:
    pickle.dump(study, f)

print(f"Optuna study saved to {study_path}")

[I 2025-11-08 13:20:14,981] A new study created in memory with name: exp_1_tfidf_log
[I 2025-11-08 13:20:16,983] Trial 0 finished with value: 0.9878986380123042 and parameters: {'max_features': None, 'ngram_min': 2, 'ngram_max': 2, 'min_df': 1, 'max_df': 0.7122770457272376, 'sublinear_tf': True, 'C': 4.688511310167415, 'solver': 'liblinear'}. Best is trial 0 with value: 0.9878986380123042.
[I 2025-11-08 13:20:20,081] Trial 1 finished with value: 0.963753804637037 and parameters: {'max_features': 5000, 'ngram_min': 1, 'ngram_max': 2, 'min_df': 4, 'max_df': 0.9429263512413611, 'sublinear_tf': False, 'C': 0.03299061697427299, 'solver': 'saga'}. Best is trial 0 with value: 0.9878986380123042.
[I 2025-11-08 13:20:30,263] Trial 2 finished with value: 0.9653613820712811 and parameters: {'max_features': 10000, 'ngram_min': 1, 'ngram_max': 4, 'min_df': 1, 'max_df': 0.789496897919448, 'sublinear_tf': False, 'C': 0.0390306919196595, 'solver': 'saga'}. Best is trial 0 with value: 0.987898638012304

Best trial value: 0.9949887521532705
Best trial params: {'max_features': 10000, 'ngram_min': 1, 'ngram_max': 4, 'min_df': 4, 'max_df': 0.7435190993013958, 'sublinear_tf': True, 'C': 9.68294408321452, 'solver': 'saga'}
Optuna study saved to experiments/exp_1_tfidf_log/optuna_study.pkl


In [16]:
# Retrain best pipeline on full training data using best params
best_params = study.best_params

# Extract tfidf params from best_params
vectorizer_best = {
    'max_features': best_params.get('max_features', None),
    'ngram_range': (best_params.get('ngram_min', 1), best_params.get('ngram_max', 2)),
    'min_df': best_params.get('min_df', 2),
    'max_df': best_params.get('max_df', 0.95),
    'sublinear_tf': best_params.get('sublinear_tf', True)
}

model_best = {
    'C': best_params.get('C', 1.0),
    'solver': best_params.get('solver', 'liblinear'),
    'max_iter': 1000,
    'random_state': 42
}

final_pipeline = Pipeline([
    ('vectorizer', TfidfVectorizer(**{k: v for k, v in vectorizer_best.items() if v is not None})),
    ('model', LogisticRegression(**model_best))
])

final_pipeline.fit(X_train_text, y_train)

# Save final pipeline and vectorizer separately
final_pipeline_path = os.path.join(experiment_dir, 'pipeline_best_full.pkl')
with open(final_pipeline_path, 'wb') as f:
    pickle.dump(final_pipeline, f)

vectorizer_path = os.path.join(experiment_dir, 'tfidf_vectorizer_best.pkl')
with open(vectorizer_path, 'wb') as f:
    pickle.dump(final_pipeline.named_steps['vectorizer'], f)

print(f"Saved final pipeline to {final_pipeline_path}")
print(f"Saved best TF-IDF vectorizer to {vectorizer_path}")

# Create final submission using final pipeline
final_test_proba = final_pipeline.predict_proba(X_test_text)[:, 1]
submission_final = pd.DataFrame({'id': test['id'], 'probability': final_test_proba})
submission_final_path = os.path.join(experiment_dir, f'{EXPERIMENT_NAME}_submission_optuna.csv')
submission_final.to_csv(submission_final_path, index=False)
print(f"Final submission saved to {submission_final_path}")

Saved final pipeline to experiments/exp_1_tfidf_log/pipeline_best_full.pkl
Saved best TF-IDF vectorizer to experiments/exp_1_tfidf_log/tfidf_vectorizer_best.pkl
Final submission saved to experiments/exp_1_tfidf_log/exp_1_tfidf_log_submission_optuna.csv


In [17]:
# Save Optuna metadata to metrics.json
metrics_path = os.path.join(experiment_dir, 'metrics.json')
with open(metrics_path, 'r') as f:
    metrics_data = json.load(f)

metrics_data['optuna_study'] = {
    'best_value': float(study.best_value),
    'best_params': study.best_params,
    'n_trials': len(study.trials),
    'study_path': study_path
}

with open(metrics_path, 'w') as f:
    json.dump(metrics_data, f, indent=2)

print(f"Updated {metrics_path} with Optuna results")

Updated experiments/exp_1_tfidf_log/metrics.json with Optuna results
